In [ ]:
# Pearson correlation
import pandas as pd
import matplotlib.pyplot as plt

# 1. Load the cleaned dataset
df = pd.read_csv('final_merged_cleaned_data.csv')

# 2. Map Severity to numerical values for mathematical analysis
# Slight = 1, Serious = 2, Fatal = 3
severity_mapping = {'Slight': 1, 'Serious': 2, 'Fatal': 3}
df['Severity_Num'] = df['Severity'].map(severity_mapping)

# 3. Identify numerical columns and calculate Pearson Correlation
# We select only numeric types (int/float)
numeric_df = df.select_dtypes(include=['number'])
correlation_matrix = numeric_df.corr(method='pearson')

# 4. Extract correlations with Severity and sort them
severity_correlations = correlation_matrix['Severity_Num'].sort_values(ascending=False)

# 5. Visualize the Importance
plt.figure(figsize=(10, 8))
# We drop 'Severity_Num' so it doesn't correlate with itself (which would be 1.0)
severity_correlations.drop('Severity_Num').plot(kind='barh', color='royalblue')
plt.title('Pearson Correlation of Features with Severity')
plt.xlabel('Correlation Coefficient (Linear Relationship)')
plt.ylabel('Features')
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

print("Correlation Coefficients with Severity:")
print(severity_correlations)

In [ ]:
# Random forest and mutual importance
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import LabelEncoder

# 1. Load the cleaned dataset
df = pd.read_csv('final_merged_cleaned_data.csv')

# 2. Prepare Target Variable (y)
# LabelEncoder turns 'Slight', 'Serious', 'Fatal' into 0, 1, 2
le = LabelEncoder()
y = le.fit_transform(df['Severity'])

# 3. Prepare Features (X)
# Drop non-predictive strings like date and time
X = df.drop(columns=['Severity', 'date', 'time'])
# Keep only numeric columns
X = X.select_dtypes(include=[np.number])
# Fill missing values with median to ensure calculations run smoothly
X = X.fillna(X.median())

# 4. Calculate Mutual Information Scores
# MI measures how much knowing a feature reduces uncertainty about the Severity
mi_scores = mutual_info_classif(X, y, random_state=42)
mi_series = pd.Series(mi_scores, name="MI Scores", index=X.columns).sort_values(ascending=False)

# 5. Calculate Random Forest Feature Importance
# This measures how useful each feature is at splitting the data to predict severity
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X, y)
rf_importance = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)

# 6. Visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))

mi_series.sort_values().plot(kind='barh', ax=ax1, color='teal')
ax1.set_title('Mutual Information Scores (Non-linear Dependency)')
ax1.set_xlabel('Score')

rf_importance.sort_values().plot(kind='barh', ax=ax2, color='darkorange')
ax2.set_title('Random Forest Feature Importance (Predictive Power)')
ax2.set_xlabel('Importance Value')

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import LabelEncoder

# 1. Load the cleaned dataset
df = pd.read_csv('final_merged_cleaned_data.csv')

# 2. Preprocessing
# Convert Severity text labels to numbers (Fatal=0, Serious=1, Slight=2)
le = LabelEncoder()
y = le.fit_transform(df['Severity'])

# Prepare features (dropping text columns and target)
X = df.drop(columns=['Severity', 'date', 'time']).select_dtypes(include=[np.number])
X = X.fillna(X.median()) # Fill any remaining NaNs with the median

# 3. Create the Train/Test Split (80% training, 20% testing)
# 'stratify=y' ensures the small number of Fatal cases are split evenly between sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 4. Initialize and Train Models
# Model A: Random Forest (using balanced weights to help with rare Fatal cases)
rf_model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)

# Model B: HistGradientBoosting (Excellent for tabular data)
hgb_model = HistGradientBoostingClassifier(random_state=42)
hgb_model.fit(X_train, y_train)

# 5. Predictions on the Test Set
y_pred_rf = rf_model.predict(X_test)
y_pred_hgb = hgb_model.predict(X_test)

# 6. PRINT RESULTS
print("Classes identified by LabelEncoder:", le.classes_)

print("\n--- RANDOM FOREST PERFORMANCE ---")
print(f"Overall Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}")
print("Confusion Matrix (Numeric):\n", confusion_matrix(y_test, y_pred_rf))
print("\nClassification Report:\n", classification_report(y_test, y_pred_rf, target_names=le.classes_))

print("\n--- HIST GRADIENT BOOSTING PERFORMANCE ---")
print(f"Overall Accuracy: {accuracy_score(y_test, y_pred_hgb):.4f}")
print("Confusion Matrix (Numeric):\n", confusion_matrix(y_test, y_pred_hgb))
print("\nClassification Report:\n", classification_report(y_test, y_pred_hgb, target_names=le.classes_))

# 7. VISUALIZE CONFUSION MATRICES
fig, ax = plt.subplots(1, 2, figsize=(14, 6))

# Plot RF Confusion Matrix
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_rf, display_labels=le.classes_, ax=ax[0], cmap='Blues')
ax[0].set_title('Random Forest (Test Set)')

# Plot HGB Confusion Matrix
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_hgb, display_labels=le.classes_, ax=ax[1], cmap='Greens')
ax[1].set_title('HistGradientBoosting (Test Set)')

plt.tight_layout()
plt.savefig('test_set_confusion_matrices.png')
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from sklearn.preprocessing import LabelEncoder

# 1. Load data
df = pd.read_csv('final_merged_cleaned_data.csv')

# 2. Merge Fatal and Serious into "Serious/Fatal"
# This creates a clearer binary signal: Minor (Slight) vs. Major (Serious/Fatal)
df['Severity_Binary'] = df['Severity'].replace({'Fatal': 'Serious/Fatal', 'Serious': 'Serious/Fatal'})

# 3. Preprocessing
le = LabelEncoder()
y = le.fit_transform(df['Severity_Binary'])

# Select numeric features (dropping IDs, text dates, and targets)
X = df.drop(columns=['Severity', 'Severity_Binary', 'date', 'time']).select_dtypes(include=[np.number])
X = X.fillna(X.median())

# 4. Stratified Split (80% Train / 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 5. Initialize Models
# Random Forest with 'balanced' weights to help the model notice the Major accidents
rf_model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
# HistGradientBoosting (Fast, modern gradient boosting)
hgb_model = HistGradientBoostingClassifier(random_state=42)

# 6. Training
rf_model.fit(X_train, y_train)
hgb_model.fit(X_train, y_train)

# 7. Visualization of Confusion Matrices
fig, ax = plt.subplots(1, 2, figsize=(14, 6))

# Plot Random Forest Matrix
ConfusionMatrixDisplay.from_predictions(
    y_test, rf_model.predict(X_test),
    display_labels=le.classes_,
    ax=ax[0],
    cmap='Blues'
)
ax[0].set_title('Random Forest: Slight vs. Serious/Fatal')

# Plot HistGradientBoosting Matrix
ConfusionMatrixDisplay.from_predictions(
    y_test, hgb_model.predict(X_test),
    display_labels=le.classes_,
    ax=ax[1],
    cmap='Greens'
)
ax[1].set_title('HistGradientBoosting: Slight vs. Serious/Fatal')

plt.tight_layout()
plt.show()

# Print Detailed Reports
print("Random Forest Report:\n", classification_report(y_test, rf_model.predict(X_test), target_names=le.classes_))
print("HGB Report:\n", classification_report(y_test, hgb_model.predict(X_test), target_names=le.classes_))

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# 1. Load Data
df = pd.read_csv('final_merged_cleaned_data.csv')

# 2. Binary Target Preparation
df['Severity_Binary'] = df['Severity'].replace({'Fatal': 'Serious/Fatal', 'Serious': 'Serious/Fatal'})

le = LabelEncoder()
y = le.fit_transform(df['Severity_Binary'])
class_names = le.classes_

# 3. Feature Selection & Scaling (Using all numeric features as per your request)
X = df.drop(columns=['Severity', 'Severity_Binary', 'date', 'time']).select_dtypes(include=[np.number])
X = X.fillna(X.median())

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 4. Define the Neural Network
mlp = MLPClassifier(hidden_layer_sizes=(64, 32),
                    activation='relu',
                    solver='adam',
                    max_iter=500,
                    early_stopping=True,
                    random_state=42)

# ==========================================
# CONSISTENT 3x5 CROSS-VALIDATION
# ==========================================
cv_repeated = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)

# Containers to ensure Part A and Part B are identical
aggregate_cm = np.zeros((2, 2))
all_y_true = []
all_y_pred = []
accuracies = []

print("Training Neural Network (3x5 Repeated CV - 15 total runs)...")

for train_idx, test_idx in cv_repeated.split(X_scaled, y):
    X_train, X_test = X_scaled[train_idx], X_scaled[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # Train the model
    mlp.fit(X_train, y_train)

    # Predict
    y_pred = mlp.predict(X_test)

    # Collect data for Part A (Averages) and Part B (Matrix)
    accuracies.append(accuracy_score(y_test, y_pred))
    all_y_true.extend(y_test)
    all_y_pred.extend(y_pred)
    aggregate_cm += confusion_matrix(y_test, y_pred, labels=[0, 1])

# --- RESULTS PART A: Averages ---
print("\n--- 3x5 Repeated Cross-Validation Results (Averages) ---")
print(f"Mean Accuracy:    {np.mean(accuracies):.4f} (+/- {np.std(accuracies):.4f})")

# --- RESULTS PART B: Detailed Breakdown ---
print("\n--- Detailed Classification Report (Aggregated 15 Folds) ---")
# This now correctly reflects all 89,505 predictions
print(classification_report(all_y_true, all_y_pred, target_names=class_names))

print("--- Confusion Matrix (Raw Numbers - Total over 15 folds) ---")
print(aggregate_cm)

# Visual Heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(aggregate_cm, annot=True, fmt='.0f', cmap='Blues',
            xticklabels=class_names,
            yticklabels=class_names)
plt.ylabel('Actual Label (Total across 15 folds)')
plt.xlabel('Predicted Label (Total across 15 folds)')
plt.title('Consistent Aggregated Confusion Matrix (3x5 CV)')
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.feature_selection import mutual_info_classif, SelectKBest

# 1. Load Data
df = pd.read_csv('final_merged_cleaned_data.csv')

# 2. Target Preparation
df['Severity_Binary'] = df['Severity'].replace({'Fatal': 'Serious/Fatal', 'Serious': 'Serious/Fatal'})
le = LabelEncoder()
y = le.fit_transform(df['Severity_Binary'])
class_names = le.classes_

# 3. Feature Selection (Top 10 via Mutual Information)
X_raw = df.drop(columns=['Severity', 'Severity_Binary', 'date', 'time']).select_dtypes(include=[np.number])
X_raw = X_raw.fillna(X_raw.median())

print("Selecting top 10 features...")
selector = SelectKBest(score_func=mutual_info_classif, k=10)
X_selected = selector.fit_transform(X_raw, y)
selected_features = X_raw.columns[selector.get_support()]
print(f"Features used: {list(selected_features)}")

# 4. Scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_selected)

# 5. Model and CV Definition
mlp = MLPClassifier(hidden_layer_sizes=(64, 32), activation='relu', solver='adam',
                    max_iter=500, early_stopping=True, random_state=42)

# This is the 3x5 strategy (15 total runs)
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)

# 6. Consistent Execution (Manual Loop)
aggregate_cm = np.zeros((2, 2))
all_y_true = []
all_y_pred = []
accuracies = []

print("\nRunning 3x5 Cross-Validation...")

for train_idx, test_idx in cv.split(X_scaled, y):
    X_train, X_test = X_scaled[train_idx], X_scaled[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    mlp.fit(X_train, y_train)
    preds = mlp.predict(X_test)

    # Store results for the final report
    all_y_true.extend(y_test)
    all_y_pred.extend(preds)
    accuracies.append(accuracy_score(y_test, preds))
    aggregate_cm += confusion_matrix(y_test, preds, labels=[0, 1])

# 7. Output Final Results
print("\n--- Final 3x5 Results ---")
print(f"Mean Accuracy: {np.mean(accuracies):.4f} (+/- {np.std(accuracies):.4f})")

print("\n--- Aggregated Classification Report (15 Folds) ---")
# This report now represents the average of all 89,505 predictions
print(classification_report(all_y_true, all_y_pred, target_names=class_names))

# 8. Consistent Visual Confusion Matrix
plt.figure(figsize=(8, 6))
sns.heatmap(aggregate_cm, annot=True, fmt='.0f', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.ylabel('Actual Label (Total across 15 folds)')
plt.xlabel('Predicted Label (Total across 15 folds)')
plt.title('Consistent Aggregated Confusion Matrix (89,505 Samples)')
plt.show()

In [ ]:
!pip install imbalanced-learn
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from imblearn.over_sampling import SMOTE  # The key library for balancing

# 1. Load Data
df = pd.read_csv('final_merged_cleaned_data.csv')

# 2. Target Preparation
df['Severity_Binary'] = df['Severity'].replace({'Fatal': 'Serious/Fatal', 'Serious': 'Serious/Fatal'})
le = LabelEncoder()
y = le.fit_transform(df['Severity_Binary'])
class_names = le.classes_

# 3. Feature Selection & Scaling
X_raw = df.drop(columns=['Severity', 'Severity_Binary', 'date', 'time']).select_dtypes(include=[np.number])
X_raw = X_raw.fillna(X_raw.median())

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

# 4. Model and CV Definition
mlp = MLPClassifier(hidden_layer_sizes=(64, 32), activation='relu', solver='adam',
                    max_iter=500, early_stopping=True, random_state=42)

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)
sm = SMOTE(random_state=42) # Initialize SMOTE

# 5. Consistent 3x5 Execution with SMOTE
aggregate_cm = np.zeros((2, 2))
all_y_true = []
all_y_pred = []
accuracies = []

print("Training Neural Network with SMOTE Balancing (15 runs)...")

for train_idx, test_idx in cv.split(X_scaled, y):
    X_train, X_test = X_scaled[train_idx], X_scaled[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # APPLY SMOTE ONLY TO TRAINING DATA
    # This creates synthetic 'Serious' cases so the model sees a 50/50 split during training
    X_train_res, y_train_res = sm.fit_resample(X_train, y_train)

    # Train on balanced data
    mlp.fit(X_train_res, y_train_res)

    # Predict on REAL, untouched test data
    preds = mlp.predict(X_test)

    # Aggregate results
    all_y_true.extend(y_test)
    all_y_pred.extend(preds)
    accuracies.append(accuracy_score(y_test, preds))
    aggregate_cm += confusion_matrix(y_test, preds, labels=[0, 1])

# 6. Final Consistent Results
print("\n--- Balanced 3x5 Results (SMOTE) ---")
print(f"Mean Accuracy: {np.mean(accuracies):.4f} (+/- {np.std(accuracies):.4f})")

print("\n--- Aggregated Classification Report (Note the improved Serious/Fatal Recall) ---")
print(classification_report(all_y_true, all_y_pred, target_names=class_names))

# 7. Visual Confusion Matrix
plt.figure(figsize=(8, 6))
sns.heatmap(aggregate_cm, annot=True, fmt='.0f', cmap='Greens',
            xticklabels=class_names, yticklabels=class_names)
plt.ylabel('Actual Label (Total across 15 folds)')
plt.xlabel('Predicted Label (Total across 15 folds)')
plt.title('Balanced Confusion Matrix (SMOTE Applied to Training)')
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from imblearn.over_sampling import ADASYN # Advanced alternative to SMOTE

# 1. Load and Prepare
df = pd.read_csv('final_merged_cleaned_data.csv')
df['Severity_Binary'] = df['Severity'].replace({'Fatal': 'Serious/Fatal', 'Serious': 'Serious/Fatal'})
le = LabelEncoder()
y = le.fit_transform(df['Severity_Binary'])
class_names = le.classes_

# 2. Features
X_raw = df.drop(columns=['Severity', 'Severity_Binary', 'date', 'time']).select_dtypes(include=[np.number])
X_raw = X_raw.fillna(X_raw.median())
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

# 3. Model - Adding a bit more complexity (more neurons) to handle the harder patterns
mlp = MLPClassifier(hidden_layer_sizes=(128, 64, 32),
                    activation='relu', solver='adam',
                    max_iter=500, early_stopping=True, random_state=42)

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)
ada = ADASYN(random_state=42)

# 4. Execution
aggregate_cm = np.zeros((2, 2))
all_y_true = []
all_y_pred = []
accuracies = []

print("Training Advanced MLP with ADASYN (15 runs)...")

for train_idx, test_idx in cv.split(X_scaled, y):
    X_train, X_test = X_scaled[train_idx], X_scaled[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # Adaptive Sampling
    X_train_res, y_train_res = ada.fit_resample(X_train, y_train)

    mlp.fit(X_train_res, y_train_res)

    # THRESHOLD TUNING: Instead of mlp.predict(), we use probabilities
    # We lower the threshold to 0.4 to catch more serious accidents without destroying precision
    probs = mlp.predict_proba(X_test)[:, 0] # Probability of class 0 (Serious/Fatal)
    preds = (probs > 0.4).astype(int)
    # Note: Depending on LabelEncoder, you may need (probs > 0.4). Check class_names order.
    # If 'Serious/Fatal' is class 0, the logic holds.

    all_y_true.extend(y_test)
    all_y_pred.extend(preds)
    accuracies.append(accuracy_score(y_test, preds))
    aggregate_cm += confusion_matrix(y_test, preds, labels=[0, 1])

# 5. Output
print("\n--- Advanced Balanced Results (ADASYN + Threshold Tuning) ---")
print(f"Mean Accuracy: {np.mean(accuracies):.4f}")
print(classification_report(all_y_true, all_y_pred, target_names=class_names))

# Plot
plt.figure(figsize=(8, 6))
sns.heatmap(aggregate_cm, annot=True, fmt='.0f', cmap='YlGnBu',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Advanced Aggregated Confusion Matrix')
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from imblearn.over_sampling import ADASYN

# 1. Load and Prepare
df = pd.read_csv('final_merged_cleaned_data.csv')
df['Severity_Binary'] = df['Severity'].replace({'Fatal': 'Serious/Fatal', 'Serious': 'Serious/Fatal'})
le = LabelEncoder()
y = le.fit_transform(df['Severity_Binary'])
class_names = le.classes_

# 2. Features
X_raw = df.drop(columns=['Severity', 'Severity_Binary', 'date', 'time']).select_dtypes(include=[np.number])
X_raw = X_raw.fillna(X_raw.median())
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

# 3. NEW ARCHITECTURE: Wide & Deep MLP
# Increasing neurons and adding 'alpha' (Regularization) to prevent overfitting
mlp = MLPClassifier(hidden_layer_sizes=(256, 128, 64),
                    activation='relu',
                    solver='adam',
                    alpha=0.001,  # L2 regularization to smooth out noise
                    learning_rate='adaptive',
                    max_iter=500,
                    early_stopping=True,
                    validation_fraction=0.1,
                    random_state=42)

cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)
ada = ADASYN(random_state=42)

# 4. Consistent 3x5 Execution
aggregate_cm = np.zeros((2, 2))
all_y_true = []
all_y_pred = []
accuracies = []

print("Training Wide & Deep MLP with ADASYN (15 runs)...")

for train_idx, test_idx in cv.split(X_scaled, y):
    X_train, X_test = X_scaled[train_idx], X_scaled[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # Adaptive Resampling
    X_train_res, y_train_res = ada.fit_resample(X_train, y_train)

    mlp.fit(X_train_res, y_train_res)

    # Prediction
    preds = mlp.predict(X_test)

    all_y_true.extend(y_test)
    all_y_pred.extend(preds)
    accuracies.append(accuracy_score(y_test, preds))
    aggregate_cm += confusion_matrix(y_test, preds, labels=[0, 1])

# 5. Output Consistent Results
print("\n--- Wide & Deep MLP + ADASYN Results ---")
print(f"Mean Accuracy: {np.mean(accuracies):.4f} (+/- {np.std(accuracies):.4f})")
print("\n--- Aggregated Classification Report ---")
print(classification_report(all_y_true, all_y_pred, target_names=class_names))

# 6. Visual Heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(aggregate_cm, annot=True, fmt='.0f', cmap='PuRd',
            xticklabels=class_names, yticklabels=class_names)
plt.ylabel('Actual Label')
plt.xlabel('Predicted Label')
plt.title('Wide & Deep Architecture Confusion Matrix (3x5 CV)')
plt.show()

In [ ]:
!pip install pytorch_tabnet
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from pytorch_tabnet.tab_model import TabNetClassifier
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from imblearn.over_sampling import ADASYN

# 1. Load and Prepare
df = pd.read_csv('final_merged_cleaned_data.csv')
df['Severity_Binary'] = df['Severity'].replace({'Fatal': 'Serious/Fatal', 'Serious': 'Serious/Fatal'})

le = LabelEncoder()
y = le.fit_transform(df['Severity_Binary'])
class_names = le.classes_

# 2. Features
X_raw = df.drop(columns=['Severity', 'Severity_Binary', 'date', 'time']).select_dtypes(include=[np.number])
X_raw = X_raw.fillna(X_raw.median())

# TabNet works best with scaled data, though it's more robust than MLP
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)

# 3. Execution Setup
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)
ada = ADASYN(random_state=42)

aggregate_cm = np.zeros((2, 2))
all_y_true = []
all_y_pred = []
accuracies = []

print("Training TabNet with ADASYN (15 runs)...")

for train_idx, test_idx in cv.split(X_scaled, y):
    X_train, X_test = X_scaled[train_idx], X_scaled[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # Adaptive Sampling for balance
    X_train_res, y_train_res = ada.fit_resample(X_train, y_train)

    # Initialize TabNet
    # n_d/n_a are the width of the network layers; 16-64 is standard.
    tabnet = TabNetClassifier(
        n_d=32, n_a=32, n_steps=5,
        optimizer_fn=torch.optim.Adam,
        optimizer_params=dict(lr=2e-2),
        scheduler_params={"step_size":10, "gamma":0.9},
        scheduler_fn=torch.optim.lr_scheduler.StepLR,
        mask_type='sparsemax', # This allows for feature selection
        verbose=0
    )

    # Fit model with internal validation to enable early stopping
    tabnet.fit(
        X_train=X_train_res, y_train=y_train_res,
        eval_set=[(X_test, y_test)],
        patience=20, max_epochs=100,
        batch_size=1024, virtual_batch_size=128,
        num_workers=0, drop_last=False
    )

    # THRESHOLD TUNING
    # Get probabilities for Class 0 (Serious/Fatal)
    probs = tabnet.predict_proba(X_test)[:, 0]
    preds = (probs > 0.4).astype(int)

    all_y_true.extend(y_test)
    all_y_pred.extend(preds)
    accuracies.append(accuracy_score(y_test, preds))
    aggregate_cm += confusion_matrix(y_test, preds, labels=[0, 1])

# 4. Output Results
print("\n--- TabNet Results (ADASYN + 0.4 Threshold) ---")
print(f"Mean Accuracy: {np.mean(accuracies):.4f}")
print(classification_report(all_y_true, all_y_pred, target_names=class_names))

# Plot Heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(aggregate_cm, annot=True, fmt='.0f', cmap='magma',
            xticklabels=class_names, yticklabels=class_names)
plt.title('TabNet Aggregated Confusion Matrix')
plt.show()